# **PRÁCTICA 1: _Construcción de un modelo URDF_**

## **_Objetivo_**
---

Construir un modelo cinemático del manipulador SO-100 a partir de las piezas STL proporcionadas, organizarlas y orientarlas en Blender, identificar las transformaciones entre sistemas de referencia y describir la estructura del robot mediante un archivo URDF. Finalmente, visualizar y verificar el modelo en VS Code mediante la extensión URDF Visualizer.

## **_Introducción_**
---

**Modelo geométrico y cinemático de un robot**

Para describir un robot manipulador en ROS, se utiliza con frecuencia el formato URDF (Unified Robot Description Format). Un archivo URDF es un documento XML que representa el robot como un conjunto de eslabones (links) conectados mediante articulaciones (joints). Cada eslabón puede incluir geometría visual, geometría de colisión y propiedades inerciales; cada junta define cómo se conecta un eslabón hijo con uno padre, el tipo de movimiento permitido y la pose relativa entre ambos sistemas de referencia.

Los tipos de junta más comunes son fixed (sin movimiento relativo), revolute (rotación limitada alrededor de un eje), continuous (rotación sin límites angulares) y prismatic (traslación). En un manipulador como el SO-100, las articulaciones de los motores se representan normalmente como articulaciones rotacionales. La etiqueta origin expresa la transformación entre los marcos de la junta padre e hijo mediante xyz (traslación) y rpy (orientación en ángulos roll, pitch y yaw, en radianes). El eje de movimiento se especifica mediante axis; por convención, se expresa en el marco de la junta.

Una descripción URDF no se obtiene únicamente colocando las mallas STL una junto a otra: es necesario determinar correctamente la relación entre los sistemas de referencia. Un error en la orientación de una junta puede hacer que el modelo se vea razonablemente bien, pero que el robot rote alrededor de un eje incorrecto. Por ello, en esta práctica se utilizará Blender para inspeccionar y preparar las piezas y, posteriormente, se describirán las transformaciones en URDF.

**Manejo básico de Blender para preparar piezas robóticas**

Blender es una herramienta de modelado 3D que permite importar mallas STL, inspeccionar su geometría, transformar objetos y establecer jerarquías padre/hijo. Cada objeto tiene un sistema de referencia local (origen del objeto y ejes locales X, Y y Z), distinto del sistema de coordenadas global de la escena. Para esta práctica, los sistemas locales deben facilitar la interpretación cinemática: el eje Z local de cada pieza asociada a un motor debe alinearse con el eje de giro de ese motor.

Las transformaciones de un objeto incluyen ubicación, rotación y escala. En Blender, Apply Rotation aplica la rotación actual a los datos de la malla y deja los valores de rotación del objeto en cero, conservando su apariencia en la escena. Esto resulta útil para preparar una geometría cuya orientación debe quedar incorporada en la malla, aunque debe hacerse deliberadamente: aplicar una rotación cambia cómo se interpretan las coordenadas locales de la geometría y puede afectar las transformaciones que luego se documenten en URDF.

En esta práctica, Blender se utilizará para construir un archivo de inspección y organización de las piezas, no para sustituir la definición cinemática del URDF. El modelo final deberá tener una jerarquía coherente y los marcos de referencia necesarios para calcular origin xyz, origin rpy y axis.



## **_Cuestionario Previo_**
---

1. ¿Cuál es la diferencia entre un link y un joint en URDF?
>En URDF, la diferencia es que el link es la parte sólida e inmóvil por sí misma (como un brazo o una rueda) que contiene la masa y el diseño visual, mientras que el joint es el mecanismo que crea el movimiento e indica cómo se mueve un link respecto al otro (si gira, se desliza o se queda fijo). En resumen: el link aporta el cuerpo físico y el joint aporta la acción y la conexión entre ellos.

2. ¿Qué información se define en las etiquetas origin, axis y parent/child de una junta URDF?
>Fijando el sistema de referencia del enlace hijo. El elemento <origin>

>Este elemento indica la ubicación y orientación del sistema de referencia L del enlace hijo. Como ya hemos comentado este sistema de referencia forma parte de la definición de la cadena cinemática y lo ubicaremos en la articulación con el enlace padre, de modo que uno de sus ejes coincida en función del tipo de articulación con el eje de giro (revolute, continuous) o con la dirección de desplazamiento (prismatic). 

>La pose del sistema de referencia L se establece en relación al sistema de referencia L del enlace padre.

>Estableciendo los enlaces de la articulación. Los elementos <parent> y <child>Estos elementos permiten proporcionar la única información que realmente es obligatoria a la hora de definir una articulación, que es la definición de los enlaces padre e hijo unidos por la misma. Indicando el eje asociado al movimiento de la articulación. El elemento <axis> Para articulaciones de tipo revolute, continuous o prismatic, el elemento <axis> permite indicar cual de los ejes del sistema de referencia L coincide con el eje de giro o de desplazamiento de la articulación.

3. ¿Cuál es la diferencia entre el sistema de coordenadas global y el sistema local de un objeto en Blender?
>La diferencia principal radica en el punto de referencia que utiliza cada sistema para medir el movimiento, la rotación y la escala de un objeto:

>• **Sistema de coordenadas global (World / Global):**

>-Tiene su origen fijo en el centro del universo 3D de la escena (el punto 0, 0, 0).

>-Los ejes X (rojo), Y (verde) y Z (azul) nunca se mueven ni cambian de orientación, sin importar hacia dónde esté volteado el objeto.

>• **Sistema de coordenadas local (Local):** 

> -Tiene su origen en el propio punto de origen del objeto (su pivote).

> -Los ejes X, Y y Z giran y se mueven junto con el objeto. Si rotas el objeto 45 grados, sus ejes locales también rotarán 45 grados

4. ¿Qué hace la operación Apply Rotation en blender?
>Estas operaciones le permiten aplicar varias transformaciones a los objetos seleccionados. Las coordenadas de transformación del objeto se transfieren a los datos del objeto. Si los objetos tienen descendientes jerárquicos, también aplica esas transformaciones a sus subordinados.

5. ¿En qué unidades se especifican las traslaciones y los ángulos en URDF?
>Las unidades de medida siguen estrictamente el Estándar del Sistema Internacional (SI):

> • **Traslaciones (\(x, y, z\)):** Se especifican en metros (m).

> • **Ángulos (rpy / roll, pitch, yaw):** Se especifican en radianes (rad).

## **_Desarrollo_**
---

**_1.  Preparación del espacio de trabajo_**

        a. Se guardo la carpeta de la practica 2 con la siguiente estructura:
```text
practica_2/
├── blender/
│   └── so100_modelo.blend
├── meshes/
│   ├── base.stl
│   └── ... (resto de las piezas proporcionadas)
├── urdf/
│   └── so100.urdf
└── imagenes/
    ├── blender_jerarquia.png
    ├── blender_ejes.png
    ├── ...
    └── urdf_visualizer.png
```
![Estructura de practica](imagenes/estructurapractica.png)

       b. Se guardo en blender un proyecto con el nombre: blender/so100_modelo.blend.
![Archivo Blender](imagenes/archivoblender.png)

        c. Se eliminaron los objetos iniciales que no se usaran.

        d. Se importaron los archivos STL mediante File → Import → STL ó arrastrándolos hacia el editor de blender desde el explorador de archivos.
![Archivos STL](imagenes/piezasimportadas.png)


**_2. Mostrar e inspeccionar los sistemas de referencia locales en Blender_**

                a. Se visualizaron los sistemas de referencia locales de cada pieza

                b. Origen local y orientación de ejes

| Pieza / eslabón | Corrección necesaria del eje "z" |
| :--- | :--- |
| **Base** | 180° |
| **Shoulder Rotation** | 180°|
| **Upper Arm** | 180° |
| **Lower Arm** | 0° |
| **Wrist Pitch Roll**| 180° |
| **Fixed Gripper**| 180° |
| **Moving Jaw**| 180° |


**_3. Orientar las piezas_**
        
        a. Se roto la pieza hasta que el eje Z local asociado al motor coincida con el eje físico de giro. Puede utilizar R seguido del eje global o local y un ángulo, o introducir valores numéricos en el panel N. Confirme visualmente la alineación desde más de una vista.

        b. Para que la orientación quede incorporada en la geometría, use Object → Apply → Rotation (Ctrl+A → Rotation) solo después de verificar el resultado. La rotación del objeto quedará en cero, pero la malla conservará la orientación visual. No aplique transformaciones de manera indiscriminada: registre qué piezas se modificaron y qué efecto tuvo en los ejes locales.

        c. Se Repitio el proceso para las piezas restantes. El eje Z del marco asociado a cada motor quedo alineado con el eje de giro correspondiente.

![Orientación](imagenes/orientacion.png)

**_4. Crear relaciones padre/hijo y obtener las transformaciones entre sistemas de referencia_**


**_5. Construir el archivo URDF en Visual Studio Code_**


**_6. Visualizar y validar el URDF con URDF Visualizer_**




## **_Análisis de Resultados_**
---

1. ¿Qué diferencias hay entre los sistemas de referencia globales y locales de Blender? ¿Cómo los utilizó para identificar los ejes de los motores?
> RESPUESTA

2. ¿Qué efecto tuvo aplicar una rotación a la malla y cómo comprobó que no se alteró la interpretación cinemática de las piezas?
> RESPUESTA

3. Explique la diferencia entre la transformación definida en joint/origin y la definida en visual/origin.
> RESPUESTA

4. ¿Qué ventajas tiene describir el robot mediante URDF frente a disponer únicamente de un conjunto de modelos STL?
> RESPUESTA

## **_Conclusiones_**
---

## **_Bibliografía_**
---